# Day 5: Silver Standardization (OMOP CDM v5.4 & Entity Linkage)

This notebook demonstrates probabilistic patient linkage using Splink and PySpark across disparate EHR records (Fellegi-Sunter model).

In [ ]:
import os
from pyspark.sql import SparkSession
from splink.spark.linker import SparkLinker
import splink.spark.comparison_library as cl

os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day05_OMOP_Linkage") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Synthetic Disparate EHR Records
data = [
    ("1", "John", "Doe", "1980-01-01", "male", "123 Main St"),
    ("2", "Jon", "Doe", "1980-01-01", "male", "123 Main Street"), # Duplicate/Match for 1
    ("3", "Jane", "Smith", "1992-05-15", "female", "456 Elm St"),
    ("4", "J.", "Smith", "1992-05-15", "female", "456 Elm St"),   # Duplicate/Match for 3
    ("5", "Robert", "Brown", "1975-10-20", "male", "789 Oak Ave")
]

columns = ["unique_id", "first_name", "last_name", "dob", "gender", "address"]
df = spark.createDataFrame(data, columns)

print("Disparate EHR Records:")
df.show()


In [ ]:
# Splink Settings for Fellegi-Sunter Linkage
settings = {
    "link_type": "dedupe_only",
    "blocking_rules_to_generate_predictions": [
        "l.dob = r.dob",
        "l.first_name = r.first_name"
    ],
    "comparisons": [
        cl.levenshtein_at_thresholds("first_name", [2]),
        cl.exact_match("last_name"),
        cl.exact_match("dob"),
        cl.exact_match("gender"),
        cl.levenshtein_at_thresholds("address", [5])
    ],
    "retain_matching_columns": True,
    "retain_intermediate_calculation_columns": True
}

# Initialize SparkLinker
linker = SparkLinker(df, settings, spark=spark)

# Estimate parameters using 'dob' and 'first_name' blocks
linker.estimate_u_using_random_sampling(max_pairs=1e6)
linker.estimate_parameters_using_expectation_maximisation("l.dob = r.dob")
linker.estimate_parameters_using_expectation_maximisation("l.first_name = r.first_name")

# Generate Predictions
predictions = linker.predict(threshold_match_probability=0.7)

print("Probabilistic Linkage Results (Match Probability > 0.7):")
predictions.as_pandas_dataframe()[[
    'match_probability', 
    'unique_id_l', 'first_name_l', 'last_name_l', 
    'unique_id_r', 'first_name_r', 'last_name_r'
]]
